# graph-vul: UniXcoder Fine-tuning on Google Colab

This notebook runs the full fine-tuning pipeline for function-level
vulnerability detection using UniXcoder, with Google Drive for persistent storage.

## Prerequisites

Before running, upload the following to **My Drive/graph-vul/data/**:

```
My Drive/
  graph-vul/
    data/
      func_subset.csv
      func_splits/
        assignments.csv
      processed/
        cwe_embeddings/
          CWE-119.pt
          CWE-125.pt
          CWE-190.pt
          CWE-20.pt
          CWE-200.pt
          CWE-416.pt
          CWE-476.pt
          CWE-787.pt
```

The `hf_cache/` directory is created automatically on first run to cache the
microsoft/unixcoder-base model weights (~500MB). Subsequent runs skip the download.

**Outputs** (written to Drive):
- `data/func_finetune_model.ckpt` — best model checkpoint (selected by val_f1)
- `data/func_finetune_metrics.json` — train/val/test metrics

In [ ]:
import subprocess

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU."
    )
print(gpu_info.stdout)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/data/graog-vul"
DRIVE_DATA = f"{DRIVE_PROJECT}/data"
DRIVE_HF_CACHE = f"{DRIVE_PROJECT}/hf_cache"

In [3]:
!mkdir -p {DRIVE_DATA}
!mkdir -p {DRIVE_HF_CACHE}

In [ ]:
from pathlib import Path

required_files = [
    "func_subset.csv",
    "func_splits/assignments.csv",
    "processed/cwe_embeddings/CWE-119.pt",
    "processed/cwe_embeddings/CWE-125.pt",
    "processed/cwe_embeddings/CWE-190.pt",
    "processed/cwe_embeddings/CWE-20.pt",
    "processed/cwe_embeddings/CWE-200.pt",
    "processed/cwe_embeddings/CWE-416.pt",
    "processed/cwe_embeddings/CWE-476.pt",
    "processed/cwe_embeddings/CWE-787.pt",
]

missing = [f for f in required_files if not Path(f"{DRIVE_DATA}/{f}").exists()]
if missing:
    raise FileNotFoundError(
        f"Missing files in {DRIVE_DATA}/:\n"
        + "\n".join(f"  - {f}" for f in missing)
    )
print(f"All {len(required_files)} required data files found.")

In [5]:
import os

os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_HF_CACHE

In [ ]:
!git clone https://github.com/noamdwc/graph-vul.git /content/graph-vul
%pip install -q pytorch-lightning transformers sentencepiece pandas numpy \
    scikit-learn torchmetrics wandb

In [ ]:
import os
import shutil
from pathlib import Path

repo_data = "/content/graph-vul/data"

if os.path.islink(repo_data):
    os.unlink(repo_data)
elif os.path.isdir(repo_data):
    shutil.rmtree(repo_data)

os.symlink(DRIVE_DATA, repo_data)
assert Path(repo_data, "func_subset.csv").exists(), "Symlink verification failed"
print(f"Symlinked {repo_data} -> {DRIVE_DATA}")

## Weights & Biases

The training script logs to WandB. Run the cell below to authenticate.
To skip WandB logging, uncomment the `WANDB_MODE` line instead.

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!wandb login

## Training Configuration

Adjust hyperparameters in the cell below. Defaults match `graph_vul/config.py`.

| Parameter | Default | Notes |
|-----------|---------|-------|
| `NUM_EPOCHS` | 10 | |
| `BATCH_SIZE` | 8 | Per-GPU batch size |
| `ENCODER_LR` | 2e-5 | UniXcoder backbone learning rate |
| `HEAD_LR` | 1e-3 | Classification head learning rate |
| `GRAD_ACCUM` | 8 | Effective batch = BATCH_SIZE x GRAD_ACCUM = 64 |
| `SEED` | 42 | |

**OOM?** Reduce `BATCH_SIZE` to 4 and increase `GRAD_ACCUM` to 16 to keep effective batch size at 64.

In [13]:
NUM_EPOCHS = 10
BATCH_SIZE = 64
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
GRAD_ACCUM = 8
SEED = 42

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

sys.path.insert(0, "/content/graph-vul")

from graph_vul.train_finetune import train

train(
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    encoder_lr=ENCODER_LR,
    head_lr=HEAD_LR,
    grad_accum=GRAD_ACCUM,
    seed=SEED,
)

In [ ]:
import json
from pathlib import Path

metrics_path = Path(DRIVE_DATA) / "func_finetune_metrics.json"
ckpt_path = Path(DRIVE_DATA) / "func_finetune_model.ckpt"

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("=== Final Metrics ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
else:
    print(f"Metrics file not found at {metrics_path}")

if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / (1024 * 1024)
    print(f"\nCheckpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
else:
    print(f"\nCheckpoint not found at {ckpt_path}")